# Notebook 05 — Resolution Sanity Test

**Purpose:** Verify that models learn terrain features, not noise or resolution artefacts.

**Approach:** Train RF baseline at 175m, 351m, and 700m resolutions. Compare metrics.  
Also test cross-resolution transfer.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else '.')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm
from scipy.ndimage import zoom

from sklearn.ensemble import RandomForestClassifier
from skimage.feature import graycomatrix, graycoprops
from scipy.stats import skew, kurtosis
import joblib

from src.utils import (
    PROCESSED_DIR, SPLITS_DIR, MODELS_DIR, FIGURES_DIR, RAW_DIR,
    TERRAIN_CLASSES, NUM_CLASSES, get_logger,
)
from src.metrics import compute_all_metrics, print_metrics_table

log = get_logger('05_resolution')

SAR_TILES_DIR = PROCESSED_DIR / 'sar_tiles'
LABEL_TILES_DIR = PROCESSED_DIR / 'label_tiles'

## 1. Prepare Multi-Resolution Tiles

If BIDR 175m swaths are available, use them. Otherwise, simulate by:
- 175m: upsample 351m tiles by 2× (or use actual BIDR if available)
- 351m: existing tiles (native resolution)
- 700m: downsample 351m tiles by 2× (area averaging)

In [ ]:
# Load split
with open(SPLITS_DIR / 'split_v1.json') as f:
    split_map = json.load(f)

# Use a well-labelled subset for this test (up to 500 tiles per split)
train_ids = [k for k, v in split_map.items() if v == 'train'][:500]
val_ids = [k for k, v in split_map.items() if v == 'val'][:200]
test_ids = [k for k, v in split_map.items() if v == 'test'][:200]

print(f"Resolution test subset: {len(train_ids)} train, {len(val_ids)} val, {len(test_ids)} test")

In [ ]:
def downsample_tile(sar_tile, factor=2):
    """Downsample by area averaging (simulates lower resolution)."""
    h, w = sar_tile.shape
    new_h, new_w = h // factor, w // factor
    reshaped = sar_tile[:new_h*factor, :new_w*factor].reshape(new_h, factor, new_w, factor)
    return reshaped.mean(axis=(1, 3))

def downsample_label(lbl_tile, factor=2):
    """Downsample labels by majority vote."""
    h, w = lbl_tile.shape
    new_h, new_w = h // factor, w // factor
    result = np.zeros((new_h, new_w), dtype=np.uint8)
    for i in range(new_h):
        for j in range(new_w):
            block = lbl_tile[i*factor:(i+1)*factor, j*factor:(j+1)*factor]
            valid = block[block != 255]
            if len(valid) > 0:
                vals, counts = np.unique(valid, return_counts=True)
                result[i, j] = vals[np.argmax(counts)]
            else:
                result[i, j] = 255
    return result

RESOLUTIONS = {
    '351m': {'factor': 1, 'tile_size': 256},
    '700m': {'factor': 2, 'tile_size': 128},
}

# Check if BIDR tiles exist for 175m
bidr_tiles = PROCESSED_DIR / 'bidr_tiles'
if bidr_tiles.exists() and any(bidr_tiles.glob('*.npy')):
    RESOLUTIONS['175m'] = {'factor': 'bidr', 'tile_size': 256}
    log.info("BIDR 175m tiles available.")
else:
    log.info("No BIDR tiles. Testing 351m and 700m only.")

print(f"Resolutions to test: {list(RESOLUTIONS.keys())}")

## 2. Extract Features at Each Resolution

In [ ]:
GLCM_DISTANCES = [1, 3, 5]
GLCM_ANGLES = [0, np.pi/4, np.pi/2, 3*np.pi/4]
GLCM_PROPS = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']

def extract_features(sar_tile):
    features = {}
    valid = sar_tile[sar_tile > 0]
    if len(valid) == 0:
        valid = sar_tile.flatten()
    features['mean'] = float(valid.mean())
    features['std'] = float(valid.std())
    features['median'] = float(np.median(valid))
    features['skewness'] = float(skew(valid))
    features['kurtosis'] = float(kurtosis(valid))
    features['p10'] = float(np.percentile(valid, 10))
    features['p90'] = float(np.percentile(valid, 90))
    
    sar_q = (np.clip(sar_tile, 0, 1) * 63).astype(np.uint8)
    glcm = graycomatrix(sar_q, distances=GLCM_DISTANCES, angles=GLCM_ANGLES,
                        levels=64, symmetric=True, normed=True)
    for prop in GLCM_PROPS:
        vals = graycoprops(glcm, prop)
        for d_idx, d in enumerate(GLCM_DISTANCES):
            features[f'{prop}_d{d}'] = float(vals[d_idx].mean())
    return features

def get_majority_label(lbl_tile):
    valid = lbl_tile[lbl_tile != 255]
    if len(valid) == 0:
        return -1
    values, counts = np.unique(valid, return_counts=True)
    return int(values[np.argmax(counts)])

In [ ]:
results_by_res = {}

for res_name, res_info in RESOLUTIONS.items():
    log.info(f"Processing {res_name}...")
    factor = res_info['factor']
    
    datasets = {}
    for split_name, tile_ids in [('train', train_ids), ('val', val_ids), ('test', test_ids)]:
        records = []
        for tid in tqdm(tile_ids, desc=f'{res_name} {split_name}', leave=False):
            sar = np.load(SAR_TILES_DIR / f"{tid}.npy")
            lbl = np.load(LABEL_TILES_DIR / f"{tid}.npy")
            
            if factor == 1:
                pass  # native resolution
            elif isinstance(factor, int) and factor > 1:
                sar = downsample_tile(sar, factor)
                lbl = downsample_label(lbl, factor)
            elif factor == 'bidr':
                # Load from BIDR tiles directory
                bidr_path = bidr_tiles / f"{tid}.npy"
                if bidr_path.exists():
                    sar = np.load(bidr_path)
                else:
                    continue
            
            label = get_majority_label(lbl)
            if label < 0:
                continue
            
            feats = extract_features(sar)
            feats['label'] = label
            records.append(feats)
        
        datasets[split_name] = pd.DataFrame(records)
    
    results_by_res[res_name] = datasets
    log.info(f"  {res_name}: train={len(datasets['train'])}, val={len(datasets['val'])}, test={len(datasets['test'])}")

## 3. Train and Evaluate RF at Each Resolution

In [ ]:
resolution_metrics = {}

for res_name, datasets in results_by_res.items():
    train_data = datasets['train']
    test_data = datasets['test']
    
    feature_cols = [c for c in train_data.columns if c != 'label']
    
    X_train = train_data[feature_cols].values
    y_train = train_data['label'].values
    X_test = test_data[feature_cols].values
    y_test = test_data['label'].values
    
    rf = RandomForestClassifier(
        n_estimators=200, class_weight='balanced',
        n_jobs=-1, random_state=42
    )
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    
    metrics = compute_all_metrics(y_test, y_pred)
    resolution_metrics[res_name] = metrics
    
    print(f"\n{'='*50}")
    print(f"Resolution: {res_name}")
    print(f"{'='*50}")
    print_metrics_table(metrics)

## 4. Cross-Resolution Transfer

In [ ]:
# Train at one resolution, test at another
res_names = list(results_by_res.keys())
transfer_matrix = pd.DataFrame(index=res_names, columns=res_names, dtype=float)

trained_models = {}
for train_res in res_names:
    train_data = results_by_res[train_res]['train']
    feature_cols = [c for c in train_data.columns if c != 'label']
    
    rf = RandomForestClassifier(
        n_estimators=200, class_weight='balanced',
        n_jobs=-1, random_state=42
    )
    rf.fit(train_data[feature_cols].values, train_data['label'].values)
    trained_models[train_res] = (rf, feature_cols)

for train_res in res_names:
    rf, feature_cols = trained_models[train_res]
    for test_res in res_names:
        test_data = results_by_res[test_res]['test']
        if len(test_data) == 0:
            continue
        X_test = test_data[feature_cols].values
        y_test = test_data['label'].values
        y_pred = rf.predict(X_test)
        acc = compute_all_metrics(y_test, y_pred)['overall_accuracy']
        transfer_matrix.loc[train_res, test_res] = acc

print("Cross-Resolution Transfer Matrix (accuracy):")
print("Rows = train resolution, Columns = test resolution")
print(transfer_matrix.to_string(float_format='%.3f'))

## 5. Summary Figure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Per-resolution metrics bar chart
res_labels = list(resolution_metrics.keys())
accs = [resolution_metrics[r]['overall_accuracy'] for r in res_labels]
mious = [resolution_metrics[r]['mean_iou'] for r in res_labels]

x = np.arange(len(res_labels))
width = 0.35
axes[0].bar(x - width/2, accs, width, label='Accuracy', color='steelblue')
axes[0].bar(x + width/2, mious, width, label='Mean IoU', color='darkorange')
axes[0].set_xticks(x)
axes[0].set_xticklabels(res_labels)
axes[0].set_ylabel('Score')
axes[0].set_title('RF Performance by Resolution')
axes[0].legend()
axes[0].set_ylim(0, 1)

# Transfer matrix heatmap
tm = transfer_matrix.astype(float).values
im = axes[1].imshow(tm, cmap='YlOrRd', vmin=0, vmax=1)
axes[1].set_xticks(range(len(res_labels)))
axes[1].set_xticklabels(res_labels)
axes[1].set_yticks(range(len(res_labels)))
axes[1].set_yticklabels(res_labels)
axes[1].set_xlabel('Test Resolution')
axes[1].set_ylabel('Train Resolution')
axes[1].set_title('Cross-Resolution Transfer (Accuracy)')
for i in range(len(res_labels)):
    for j in range(len(res_labels)):
        if not np.isnan(tm[i, j]):
            axes[1].text(j, i, f'{tm[i,j]:.2f}', ha='center', va='center', fontsize=12)
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'resolution_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

print("\nInterpretation:")
print("  - Graceful degradation with resolution → model learns terrain morphology")
print("  - Catastrophic drop → model overfits to resolution-specific noise")
print("  - Identical performance → features are scale-robust")